# Solar filament instance segmentation: the whole pipeline

This notebook runs our final pipeline on the 180 test images, writes `submission.csv`, and shows each stage on two of them.

**Inputs.** Attach the competition data (`filament-segmentation-2026`) and turn Internet on: the notebook clones this repository at its release tag `v1.1` (`"ms125m4"` and `"ms125m4r4"` were added after that tag: to run them, set `SOURCE_TAG = "origin/final-ms125"` in the first code cell). The final `MODE = "ms125"` and `MODE = "pcombo"` need only the competition data: their weights and stored maps are in the repository. The other later submissions, except `"ms125m4"` and `"ms125m4r4"`, also need the release dataset (stored maps and models; private, not published). Every file is hash-checked when it loads; nothing is trained.

**Runtime.** The final mode builds its CSV from the stored maps on CPU in about 9 min, plus installing the pinned environment. `"pcombo"` runs the networks first (about 14 min on one T4).

**Check.** Part B ends by printing whether `submission.csv` is byte-identical to our submission for `MODE`.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Clone this repository at its release tag (needs Internet), unless this notebook already runs inside a checkout.
SOURCE_TAG = "v1.1"
if not (REPO / "notebooks/reproduce_maps.ipynb").is_file():
    REPO = Path("/kaggle/working/filament-reproduction-source")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Matapanino/filament-segmentation-2026-solution.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_TAG], check=True)
os.chdir(REPO / "notebooks")  # the stage notebooks find the checkout from their working directory
sys.path.insert(0, str(REPO / "src"))  # the stage notebooks install the package; this notebook reads its mode table and figures

## Which CSV

`MODE` in the next cell chooses which of our submissions to rebuild. The default is the final one, `"ms125"`. The first eight modes read the same probability maps from the networks in this repository; they differ in the instance filter and the mask repair (stages 4 and 5 below), and `"mftta"` also uses 8-view Mask2Former maps. The last twelve (from `"ms125"` on) are our later submissions (including `"ms125m4"`, the final submission with a four-classifier instance filter, and `"ms125m4r4"`, which also averages four regressors for the repair threshold): they read stored probability maps of stronger Mask2Former models (5 cross-validation fold models each, several input scales and views) and the mean of the 5 fold U-Nets (`"ms125"`, `"ms125m4"` and `"ms125m4r4"` from `weights/final/` in this repository, the others from the private release dataset), with a Mask2Former weight of 0.55 in the averaged map. The names are cumulative: "Refit: + context + neighbours" is the filter refitted with the base, context and neighbour features.

| What it builds | `MODE` |
| --- | --- |
| First filter, no repair | `"equivalent"` |
| First filter, one-threshold repair | `"pnat"` |
| First filter, per-object repair | `"pcand"` |
| Refit: + context | `"fresh_ctrl"` |
| Refit: + context + neighbours | `"pneigh"` |
| Refit: + context + Mask2Former shape | `"mfrich"` |
| Refit: all four groups (needs only the competition data) | `"pcombo"` |
| All four groups + 8-view Mask2Former (second selection) | `"mftta"` |
| Crop-1536 Mask2Former models, 8 views + 1.25× scale (2 views); refitted filter and repair; 5-fold U-Nets **(final)** | `"ms125"` |
| Crop-1536 Mask2Former, 8 views + 1.25× scale; final filter averaged with LightGBM, XGBoost and CatBoost; refitted repair; 5-fold U-Nets | `"ms125m4"` |
| Crop-1536 Mask2Former, 8 views + 1.25× scale; four-classifier filter; repair threshold averaged over four regressors; 5-fold U-Nets | `"ms125m4r4"` |
| Crop-1536 Mask2Former models, 8 views + 1.25× scale (8 views); refitted filter and repair; 5-fold U-Nets | `"ms125d4"` |
| Crop-1536 Mask2Former models, 8 views + 1.25× (8 views) + 1.5× (4 views); refitted filter and repair; 5-fold U-Nets | `"ms4"` |
| Crop-1536 Mask2Former models, 8 views; refitted filter and repair; 5-fold U-Nets | `"f1c15u"` |
| Crop-768 + crop-1024 Mask2Former models, 8 views; refitted filter; 5-fold U-Nets | `"c7c10d4_un5"` |
| Crop-768 + crop-1024 Mask2Former models; refitted filter and repair; 5-fold U-Nets | `"c7c10_refit_un5"` |
| Crop-768 + crop-1024 Mask2Former models; refitted filter; 5-fold U-Nets | `"c7c10_un5"` |
| Crop-768 + crop-1024 Mask2Former models; refitted filter | `"c7c10"` |
| Crop-768 Mask2Former models; refitted filter | `"c768"` |
| Crop-768 Mask2Former models, Mask2Former weight 0.25; refitted filter | `"c768_w25"` |

In [ ]:
MODE = None  # None = the final submission (modes.DEFAULT_MODE, the "(final)" row); or any MODE in the table above

## Part A (GPU): stage 1, probability maps

Three networks give every pixel a filament probability. They read the image at different scales, so their errors differ:

- the **full-disk U-Net** reads the whole disk resized to 1024 px, for large-scale context;
- the **local U-Net** reads native-resolution 1024 px tiles with raw, contrast-enhanced (CLAHE) and detail channels, for thin structure;
- **Mask2Former** predicts whole filament instances; in this GPU stage (the earlier pipeline, `"pcombo"` and the modes before it) it runs as three models, one per training fold, whose maps are averaged. The final `"ms125"` instead reads the stored maps of five fold models (see below).

The networks run in deterministic mode (fixed seeds, deterministic CUDA kernels, a fixed image order), so the map archives are byte-identical from run to run. If a completed output of this stage is attached, it is skipped.

**Stage 1b, only for `MODE = "mftta"` (our second selection), on GPU.** Each Mask2Former model also reads the image in 8 views (flips and right-angle rotations, original included); each output is turned back and the 8 maps are averaged. This takes about another 42 min on one T4 and is skipped when its output is attached.

**Final submission (`MODE = "ms125"`).** Its CSV is built from the probability maps stored in this repository (`weights/final/`), so it needs no GPU. On a T4, this stage first re-reads two test images with the stored models (five Mask2Former and ten U-Net fold models) and prints how closely the new maps match the stored ones (`reproduce_chain_maps.ipynb`, a few minutes).

In [ ]:
# Stage 1 (GPU): the GPU notebooks MODE still needs. A stage is skipped when its output is attached or already in /kaggle/working/maps.
from filseg.reproduction.paths import gpu_stages
MODE, GPU_STAGES = gpu_stages(MODE)
print("MODE:", MODE, "- GPU stages to run:", GPU_STAGES or "none (their outputs are attached)")
if "reproduce_maps.ipynb" in GPU_STAGES:
    get_ipython().run_line_magic("run", "reproduce_maps.ipynb")
if "reproduce_mf_views.ipynb" in GPU_STAGES:
    get_ipython().run_line_magic("run", "reproduce_mf_views.ipynb")
if "reproduce_chain_maps.ipynb" in GPU_STAGES:
    get_ipython().run_line_magic("run", "reproduce_chain_maps.ipynb")

## Part B (CPU): stages 2–6, from maps to the submission CSV

2. **Averaged map.** For the final `"ms125"`: p = 0.55 · Mask2Former + 0.225 · full + 0.225 · local, where Mask2Former is the mean of the stored maps of five fold models (8 views and a 1.25× zoom each) and each U-Net map is the mean of five fold models. For the earlier pipeline (`"pcombo"` and the modes before it): p = 0.375 · full + 0.375 · local + 0.25 · Mask2Former (mean of three fold models).
3. **Candidates.** The map threshold, closing, a linker for collinear tips and an area limit cut p into non-overlapping candidates.
4. **Instance filter.** A logistic regression on features from the probability maps and the raw image keeps a candidate if its probability of matching a filament reaches the keep threshold.
5. **Mask repair.** Each kept mask is redrawn from p at a repair threshold predicted for that mask.
6. **CSV.** Each mask is run-length encoded. The notebook checks the format (every row decodes, no empty or overlapping masks, unique ids) and prints whether the CSV equals our submission.

In [ ]:
# Stage 2 (CPU work): the submission CSV for MODE.
get_ipython().run_line_magic("run", "reproduce_submission.ipynb")

## What each stage did, on two test images

The cell below reruns Part B on two test images with the same `MODE`, checks that every intermediate result has the same hash as in the CSV run above, and draws it. It reads the CSV run's records and never changes the CSV. The text under the figures explains each panel.

In [ ]:
from IPython.display import Image, display
from filseg.reproduction.figures import illustrate
FIGURE_IMAGES = ['20140420195854Bh', '20111122063134Lh']  # two test images: the most dropped candidates, and the largest change from the first filter
for key in FIGURE_IMAGES:
    png, summary = illustrate(key, IMAGE_ROOT, MAPS_ROOT, OUTPUT, MODE, variant_maps=VARIANT_MAPS)
    print(summary)
    display(Image(data=png))

**Reading the figures** (top row, then bottom row, left to right):

1. **Test image.** One full-disk H-alpha image; filaments are the thin dark features on the disk.
2. **Full-disk U-Net.** Probability map of the network that sees the whole disk at half resolution.
3. **Local U-Net.** Probability map of the network that sees native-resolution tiles; look for sharper thin parts.
4. **Mask2Former.** Mean map of the fold models of `MODE` (five, each in 8 views and at 1.25× zoom, for the final `"ms125"`; three for the earlier pipeline); it marks some structures the U-Nets leave out.
5. **Averaged map (stage 2).** Structures that only one network sees are damped; structures the networks agree on stay bright.
6. **Instance filter (stage 4): kept (yellow) / dropped (grey).** Each coloured region is one candidate of stage 3.
7. **Submitted masks.** One colour per mask; each mask is one row of the CSV.
8. **Mask repair (stage 5): added (sky blue) / removed (reddish purple) pixels.** It explains panel 7: the repair moves outlines by a thin rim and never adds, removes or merges masks.